In [ ]:
# Goals
# 1. Impact Weighting posts by engagement
# 2. Cross-stock sntiment infuence
# 3. Differeniation between generic and stock-specific sentiment signals
# 4. Compare FinBERT vs CryptoBERT vs DeepSeek
# 5. Market Regimes influence - volatility, bull vs bear, crisis vs normal
# 6. Sentiment Esemble model vs DDQN vs Baselines
# 7. Ablation Study - removing one component at a time to see its impact on performance (with/without sentiment, with/without technical indicators)
# 8. Posts count per day influence (100vs10)
# 9. Compare WinRate, SharpeRatio, Cumulative Returns


In [ ]:
import pandas as pd
import numpy as np
from collections import deque
from tensorflow.keras.models import Sequential, clone_model
from tensorflow.keras.layers import Dense, InputLayer, Dropout
from tensorflow.keras.optimizers import Adam
from sklearn.preprocessing import OneHotEncoder, MinMaxScaler
import random
import time 
import tensorflow as tf
from Data_Preprocessing import Data_Preprocessing
from Calculate_Returns import Calculate_Returns
from Triple_Barrier_Labelel import Triple_Barrier_Labelel
from Model_Train import Model_Train
import torch
import os
import gc
from Association_Rule_Mining import create_tweet_dataset, create_continous_dataset, create_triple_barrier_labeling, create_categorize_dataset
from math import sqrt
import matplotlib.pyplot as plt

DEBUG = False

dp = Data_Preprocessing()
mt = Model_Train('', '')
encoder = OneHotEncoder(sparse=False)
scaler = MinMaxScaler()
tbl = Triple_Barrier_Labelel()

MEMORY_LENGTH = 100
BATCH_SIZE = 64
MODEL_DESIGN = "64/64"
LEARNING_RATE = 0.0005
FEE = 0.01
INITIAL_CASH = 100000
TARGET_UPDATE = 20
EPISODES = 50
EPSILON_MIN = 0.01
EPSILON_DECAY = 0.95
GAMMA = 0.95
EPSILON = 1.0
TWEETS_RANDOM_SAMPLE = True
TWEETS_ENGAGEMENT_POSTS = False
TWEETS_DAILY_SAMPLE_SIZE = 50
TWEETS_RANDOM_SEED = 42
TEST_REPEATS = 5

STARTING_DATE = "2018-01-01"
ENDING_DATE ="2021-01-01"

device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')

In [ ]:
def evaluate_model(original_data, scaled_data, mod, log = False, market = "BTC"):
    print("Evaluating...")
    returns = Calculate_Returns(INITIAL_CASH, FEE, pd.Series([x[0] for x in original_data]), pd.Series(dtype="float64"), pd.Series(dtype="float64"), pd.Series(dtype="float64"))
    profits = []
    action = None

    for step, el in enumerate(original_data[:-1]):
        print(step)
        offset = step
        q_values = mod.predict(np.array(scaled_data[offset]).reshape(1,-1), verbose=0)
        action = np.argmax(q_values[0])
        offset += 1
        if log:
            print(f"Step: {offset}/{len(original_data[:-1])}, Prev Price: {original_data[offset - 1][0]}, Selected Action: {action}")
        delta = ((original_data[offset][0] - original_data[offset - 1][0]) / original_data[offset - 1][0])        
        if action == 0: #Hold
            reward = -0.01
        elif action == 1: #Short
            reward = -delta
        else: #Long
            reward = delta
        returns.perform_action(action, offset)
        if log:
            print(f"Next Price: {original_data[offset][0]}, Reward: {reward}\n")
        profits.append(reward)
    positive_rewards = (len([p for p in profits if p > 0])/len(profits))*100
    if len(returns.records):
        wins = (len([e['PnL'] for e in returns.records if e['PnL'] > 0])/len(returns.records))*100
        sum_pnl = sum([e['PnL'] for e in returns.records])
        days = 365 if market == "BTC" else 252
        sharpe_ratio = returns.sharpe(days)

    if log and len(returns.records):
        print(f"Positive Rewards {positive_rewards}")
        print(f"Wins {wins}")
        print(f"Sum PnL {sum_pnl}")
        print(f"Sharpe Ratio {sharpe_ratio}")
        
    return {
        "Positive Rewards": positive_rewards,
        "Wins": wins if len(returns.records) else 0,
        "Sum PnL": sum_pnl if len(returns.records) else 0,
        "Sharpe Ratio": sharpe_ratio if len(returns.records) else 0
    }

In [ ]:
class Agent():
    def __init__(self, action_size, state_size, gamma, epsilon, epsilon_min, epsilon_decay):
        self.action_size = action_size
        self.state_size = state_size
        self.gamma = gamma
        self.epsilon = epsilon
        self.epsilon_min = epsilon_min
        self.epsilon_decay = epsilon_decay
        self.memory = deque(maxlen=MEMORY_LENGTH)
        self.model = self.create_model()
        self.target_model = clone_model(self.model)
        self.optimizer = Adam(learning_rate=LEARNING_RATE) 
        self.step_counter = 0
        
    def generate_model(self, state_size, action_size):
        layers = MODEL_DESIGN.split("/")
        model = Sequential()
        state_size = state_size
        model.add(InputLayer(input_shape=(state_size,), name="InputLayer"))
        for index, units in enumerate(layers):
            model.add(Dense(units=units, activation="relu", name=f"HiddenLayer{index}"))
            model.add(Dropout(0.1))
        model.add(Dense(units=action_size, activation='linear', name="OutputLayer"))
        return model

    def create_model(self):
        return self.generate_model(self.state_size, self.action_size)

    def act(self, state):
        if random.uniform(0,1) < self.epsilon:
            rand_action = random.randrange(self.action_size)
            return rand_action
        state = np.array(state).reshape(1, -1)
        q_values = self.model(np.array(state, dtype=np.float32), training=False).numpy()
        return np.argmax(q_values[0])
    
    def remember(self, state, action, reward, new_state, done):
        self.memory.append((state, action, reward, new_state, done))
        
    def update_target_model(self):
        self.target_model.set_weights(self.model.get_weights())
    
    def replay(self):
        if len(self.memory) < BATCH_SIZE:
            return
        minibatch = random.sample(self.memory, BATCH_SIZE)
        states, actions, rewards, new_states, done = zip(*minibatch)
        states = np.array(states, dtype=np.float32)
        new_states = np.array(new_states, dtype=np.float32)
        actions = np.array(actions, dtype=np.int32)
        rewards = np.array(rewards, dtype=np.float32)
        done = np.array(done, dtype=np.float32)

        best_action_indices = np.argmax(self.model(new_states, training=False).numpy(), axis=1)
        target = rewards + (1 - done) * self.gamma * self.target_model(new_states, training=False).numpy()[np.arange(BATCH_SIZE), best_action_indices]
        with tf.GradientTape() as tape:
            current_Q_values = self.model([states], training=True)
            action_mask = tf.one_hot(actions, current_Q_values.shape[1])
            predicted_Q_values = tf.reduce_sum(current_Q_values * action_mask, axis=1)
            loss = tf.keras.losses.MeanSquaredError()(target, predicted_Q_values)
        gradients = tape.gradient(loss, self.model.trainable_variables)
        self.optimizer.apply_gradients(zip(gradients,self.model.trainable_variables))
        self.step_counter += 1
        if self.epsilon > self.epsilon_min:
            self.epsilon *= self.epsilon_decay
        if self.step_counter % TARGET_UPDATE == 0:
            self.step_counter = 0
            self.update_target_model()
        del states, actions, rewards, new_states, done, minibatch
        gc.collect()

In [ ]:
class Environment():
    def __init__(self, data, data_scaled):
        self.action_size = 3
        self.state_size = np.array(data_scaled).shape[1]
        self.data = data
        self.data_scaled = data_scaled
        self.offset = 0
        self.steps = len(data)
        self.returns = Calculate_Returns(INITIAL_CASH, FEE, pd.Series([x[0] for x in data]), pd.Series(dtype="float64"), pd.Series(dtype="float64"), pd.Series(dtype="float64"))
    
    def _append_action_and_position_to_state(self, state, action):
        one_hot_action = np.zeros(self.action_size)
        if action is not None:
            one_hot_action[action] = 1
        position_one_hot = np.zeros(3)
        position_index = int(self.returns.context)
        position_one_hot[position_index] = 1
        return np.concatenate([state, one_hot_action, position_one_hot])
    
    def step(self, action):
        self.offset = self.offset + 1
        new_state = self.data_scaled[self.offset]
        delta = ((self.data[self.offset][0] - self.data[self.offset - 1][0]) / self.data[self.offset - 1][0])
        if action == 0: #Hold
            reward = -0.01
        elif action == 1: #Short
            reward = -delta
        else: #Long
            reward = delta
        self.returns.perform_action(action, self.offset)
        done = self.offset == len(self.data) - 1
        return new_state, reward, done
    
    def reset(self):
        self.offset = 0
        self.returns = Calculate_Returns(INITIAL_CASH, FEE, pd.Series([x[0] for x in self.data]), pd.Series(dtype="float64"), pd.Series(dtype="float64"), pd.Series(dtype="float64"))
        state_only = self.data_scaled[self.offset]
        return state_only

In [ ]:
class DQNAlgorithm():
    def __init__(self, data, data_scaled, episodes, gamma, epsilon, epsilon_min, epsilon_decay):
        self.episodes = episodes
        self.data = data
        self.data_scaled = data_scaled
        self.env = Environment(data, data_scaled)
        self.steps = self.env.steps
        self.agent = Agent(self.env.action_size, self.env.state_size, gamma, epsilon, epsilon_min, epsilon_decay)
        self.best_model_win_rate = 0
        encoder.fit([[0], [1], [2]])
        self.best_models = []
        
    def debug(self):
        print("Debugging...")
        state = self.env.reset()
        profits = []
        for step in range(self.steps):
            action = self.agent.act(state)
            new_state, reward, done = self.env.step(action)
            print(f"Step: {step}/{self.steps}, Prev Price: {self.data[step][0]}, Previous Context: {self.env.returns.context}, Selected Action: {action}")
            print(f"Next Price: {self.data[step + 1][0]}, Reward: {reward}\n")
            self.agent.remember(state, action, reward, new_state, done)
            profits.append(reward)
            if done == True:
                pos_rewards = (len([p for p in profits if p > 0])/len(profits))*100
                if pos_rewards > self.best_model_win_rate:
                    self.best_model_win_rate = pos_rewards
                    self.best_model = self.agent.model
                print(f"Positive Rewards: {pos_rewards} %")
                if len(self.env.returns.records):
                    print(f"Wins: {(len([e['PnL'] for e in self.env.returns.records if e['PnL'] > 0])/len(self.env.returns.records))*100} %")
                    print(f"Sum PnL: {sum([e['PnL'] for e in self.env.returns.records])}")
                print(f"Sharpe Ratio: {self.env.returns.sharpe()}")
                break
            self.agent.replay()
            state = new_state

    def run(self):
        print("Training...")
        for episode in range(self.episodes):
            state = self.env.reset()
            actions = 0
            start_time = time.time()
            for step in range(self.steps):
                actions+=1
                action = self.agent.act(state)
                new_state, reward, done = self.env.step(action)
                self.agent.remember(state, action, reward, new_state, done)
                if done == True:
                    new_model = clone_model(self.agent.model)
                    new_model.set_weights(self.agent.model.get_weights())
                    self.best_models.append(new_model)
                    print(f"Episode {episode + 1}/{self.episodes}")
                    end_time = time.time()
                    print(f"Elapsed time: {end_time - start_time}\n")
                    break
                self.agent.replay()
                state = new_state
        print("Cleaning memory...")
        gc.collect()
        tf.keras.backend.clear_session()
        return self.best_models

In [ ]:
cases = pd.read_csv("TestCases/DDQN_Test_Cases.csv")
for test_id in range(20, len(cases)):
    params = cases["Parameters"][test_id]
    market = cases["Market"][test_id]
    name = cases["TestCaseName"][test_id]
    
    continous_df = create_continous_dataset(market, starting_date=STARTING_DATE, ending_date=ENDING_DATE)
    continous_df_with_tbl = create_triple_barrier_labeling(continous_df)
    tweets_df = create_tweet_dataset("Datasets/investing_classified_sentiments" if market != "BTC-USD" else "Datasets/btc_classified_sentiments") #"Datasets/investing_classified_sentiments"
    merged_df = pd.concat([continous_df_with_tbl, tweets_df], axis=1).dropna()
    # extended = merged_df #extend_time_columns(merged_df, skip_cols=["signals"], t=7)
    categorized_for_trade_profitable = create_categorize_dataset(merged_df, suffix_vals=['bearish', 'Bearish', 'bullish', 'Bullish', '1.0', '1', '0.0', '0', '-1', '-1.0'], skip_cols=["next_day_label", "signals", "previous_label"])
    merged_df = pd.concat([merged_df, categorized_for_trade_profitable], axis=1).dropna()

    sel_param = ["close"] + [el.strip() for el in params.split(",")]
    
    existing_cols = [col for col in sel_param if col in merged_df.columns]
    merged_data = merged_df[existing_cols]
    
    merged_data = merged_data.values.tolist()
    train_end = int(len(merged_data) * 0.8)
    train_data = merged_data[:train_end]
    test_data  = merged_data[train_end:]
    train_x = np.array(train_data)[:,1:]
    train_y = np.array(train_data)[:,:1]
    test_x = np.array(test_data)[:,1:]
    test_y = np.array(test_data)[:,:1]
    scaler.fit(train_x)
    train_scaled = scaler.transform(train_x).tolist()
    test_scaled = scaler.transform(test_x).tolist()
    
    test_results_df = []
    
    for rep in range(0, TEST_REPEATS):

        dqn = DQNAlgorithm(data=train_y, data_scaled=train_scaled, episodes=EPISODES, gamma=GAMMA, epsilon=EPSILON, epsilon_min=EPSILON_MIN, epsilon_decay = EPSILON_DECAY)
        if DEBUG == True:
            dqn.debug(train_data)
            break
        else:
            best_models = dqn.run()

        test_results = {
            "Positive Rewards": -np.inf,
            "Wins": -np.inf,
            "Sum PnL": -np.inf,
            "Sharpe Ratio": -np.inf
        }
        
        for index, model in enumerate(best_models):
            score = evaluate_model(test_y, test_scaled, model, False, market)
            if score["Positive Rewards"] > test_results["Positive Rewards"]:
                test_results = score

        test_results_df.append(test_results)

    test_results_df = pd.DataFrame(test_results_df)
    columns = ["TestCaseName", "Test Repeats", "Train Episodes", "Market", "Parameters", "Positive Rewards", "Wins", "Sum PnL", "Sharpe Ratio"]
    test_conc = [name, TEST_REPEATS, EPISODES, market, params, f"Mean: {test_results_df['Positive Rewards'].mean()}, Std: {test_results_df['Positive Rewards'].std()}", f"Mean: {test_results_df['Wins'].mean()}, Std: {test_results_df['Wins'].std()}", f"Mean: {pd.to_numeric(test_results_df['Sum PnL'], errors='coerce').mean()}, Std: {pd.to_numeric(test_results_df['Sum PnL'], errors='coerce').std()}", f"Mean: {test_results_df['Sharpe Ratio'].mean()}, Std: {test_results_df['Sharpe Ratio'].std()}"]
    pd.DataFrame([test_conc], columns=columns).to_csv(f"TestResults/DDQN_Test_Results.csv", mode="a", index=False, header=not os.path.exists("TestResults/DDQN_Test_Results.csv"))
